# mcPHASES EDA (aggregates only)

**Run this yourself.** Claude is denied this notebook and the data folder (D18, D19, D23).

- Every section prints plain-text tables of aggregates: counts, % missing, percentiles, and
  categories with small groups shown as `<5`. Read them here, and paste them back as they are.
- Never add cells that print rows, `df.head()`, ids, or one participant's values, and never put a
  single participant's series in slides or the writeup.
- Outputs are stripped on commit (nbstripout).
- Sections 1–5 describe the data. Section 6 is the evidence behind the cleaning rules in
  `docs/data.md`: rerun it on a new dataset version and update the doc if anything moves.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "signals").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from signals.mcphases import data_dir, load  # noqa: E402 (needs ROOT on sys.path)

DATA = data_dir()
print("Reading from:", DATA)
pd.set_option("display.width", 200)

## Helpers

Suppression and summary functions used by every section.

In [ ]:
MIN_CELL = 5  # a category seen in fewer participants than this is shown as "<5"
MAX_LEVELS = 15  # more distinct values = free text or id-like: count only, never the values
VALUE_PCTS = [0.01, 0.25, 0.5, 0.75, 0.99]  # no min/max: an extreme is one person's value
PEOPLE_PCTS = [0.1, 0.25, 0.5, 0.75, 0.9]  # spread across ~42 participants

HORMONES = ["lh", "estrogen", "pdg"]
SELF_REPORT = [
    "flow_volume",
    "flow_color",
    "appetite",
    "exerciselevel",
    "headaches",
    "cramps",
    "sorebreasts",
    "fatigue",
    "sleepissue",
    "moodswing",
    "stress",
    "foodcravings",
    "indigestion",
    "bloating",
]
# file -> (day column, value columns): signals and activity that line up with the Google Health API
SIGNALS = {
    "computed_temperature": (
        "sleep_end_day_in_study",
        [
            "nightly_temperature",
            "temperature_samples",
            "baseline_relative_sample_sum",
            "baseline_relative_nightly_standard_deviation",
        ],
    ),
    "resting_heart_rate": ("day_in_study", ["value", "error"]),
    "heart_rate_variability_details": (
        "day_in_study",
        ["rmssd", "coverage", "low_frequency", "high_frequency"],
    ),
    "respiratory_rate_summary": (
        "day_in_study",
        ["full_sleep_breathing_rate", "deep_sleep_breathing_rate", "full_sleep_signal_to_noise"],
    ),
    "sleep": ("sleep_end_day_in_study", ["minutesasleep", "timeinbed", "efficiency"]),
    "steps": ("day_in_study", ["steps"]),
    "active_minutes": ("day_in_study", ["sedentary", "lightly", "moderately", "very"]),
    "exercise": ("start_day_in_study", ["duration", "activeduration", "averageheartrate", "steps"]),
    "active_zone_minutes": ("day_in_study", ["total_minutes"]),
    "time_in_heart_rate_zones": (
        "day_in_study",
        ["below_default_zone_1", "in_default_zone_1", "in_default_zone_2", "in_default_zone_3"],
    ),
}
# One column at a time, binned, never crossed. income and sexually_active are not reported.
DEMOGRAPHICS = ["ethnicity", "self_report_menstrual_health_literacy"]
FILES = ["hormones_and_selfreport", "subject-info", *SIGNALS]
# file -> the key that should identify one row (section 6). Timestamps are time of day only,
# so every key includes the day column.
DUP_KEYS = {
    "hormones_and_selfreport": ["id", "day_in_study"],
    "resting_heart_rate": ["id", "day_in_study"],
    "respiratory_rate_summary": ["id", "day_in_study", "timestamp"],  # naps: several per day
    "heart_rate_variability_details": ["id", "day_in_study", "timestamp"],
    "computed_temperature": ["id", "sleep_start_day_in_study", "sleep_start_timestamp"],
    "sleep": ["id", "sleep_start_day_in_study", "sleep_start_timestamp"],
    "steps": ["id", "day_in_study", "timestamp"],
    "active_minutes": ["id", "day_in_study"],
    "exercise": ["id", "start_day_in_study", "start_timestamp"],
    "active_zone_minutes": ["id", "day_in_study", "timestamp", "heart_zone_id"],
    "time_in_heart_rate_zones": ["id", "day_in_study"],
}
ZONES = ["below_default_zone_1", "in_default_zone_1", "in_default_zone_2", "in_default_zone_3"]


def show(title, df):
    """Print a table as plain text: easy to read here and to paste back."""
    print(f"\n=== {title} ===")
    print(df.to_string() if len(df) else "(empty)")


def pct_missing(values):
    return round(100 * float(values.isna().mean()), 1)


def numeric(values):
    """n, % missing, and 1st to 99th percentiles of one column."""
    nums = pd.to_numeric(values, errors="coerce")
    row = {"n": int(nums.notna().sum()), "% missing": pct_missing(values)}
    for q in VALUE_PCTS:
        row[f"p{round(q * 100)}"] = round(float(nums.quantile(q)), 3) if row["n"] else None
    return row


def across_people(per_person):
    """Spread of a per-participant count across participants (never the per-person list)."""
    row = {"participants": int(per_person.size)}
    for q in PEOPLE_PCTS:
        row[f"p{round(q * 100)}"] = (
            round(float(per_person.quantile(q)), 1) if per_person.size else None
        )
    return row


def categories(df, col):
    """Rows and participants per value. Small groups show as <5; free text is never shown."""
    n_unique = df[col].nunique(dropna=True)
    if n_unique > MAX_LEVELS:
        return pd.DataFrame([{"value": f"({n_unique} distinct values, not shown)"}])
    labels = df[col].astype("object").where(df[col].notna(), "<missing>").astype(str)
    rows = labels.value_counts()
    people = df.groupby(labels)["id"].nunique() if "id" in df else rows
    out = []
    for value, n_rows in rows.items():
        small = people[value] < MIN_CELL
        out.append(
            {
                "value": value,
                "rows": f"<{MIN_CELL} ppl" if small else int(n_rows),
                "participants": f"<{MIN_CELL}" if small else int(people[value]),
            }
        )
    return pd.DataFrame(out)


def suppress(count):
    return count if count >= MIN_CELL else f"<{MIN_CELL}"


def day_range(df, day_col):
    """Where each study interval sits on the day axis: 1st / 50th / 99th percentile."""
    days = pd.to_numeric(df[day_col], errors="coerce")
    return {
        str(k): f"{v.quantile(0.01):.0f} / {v.quantile(0.5):.0f} / {v.quantile(0.99):.0f}"
        for k, v in days.groupby(df["study_interval"])
    }


def try_load(name, columns=None):
    """load() from DATA, or None (with a note) if the file isn't downloaded."""
    try:
        return load(name, columns, DATA)
    except FileNotFoundError:
        print(f"({name}: file not found)")
        return None


def shift_days(s, offset):
    """Re-index an (id, day) series so that day d holds the value from day d + offset."""
    out = s.copy()
    out.index = pd.MultiIndex.from_arrays(
        [s.index.get_level_values(0), s.index.get_level_values(1) - offset]
    )
    return out

## 1. Inventory: files, rows, participants, columns, % missing

In [ ]:
summary, columns = [], []
for name in FILES:
    try:
        df = load(name, root=DATA)
    except FileNotFoundError:
        summary.append({"file": name, "rows": "file not found", "participants": "", "columns": ""})
        continue
    by_interval = (
        {str(k): suppress(v) for k, v in df["study_interval"].value_counts().items()}
        if "study_interval" in df
        else ""
    )
    summary.append(
        {
            "file": name,
            "rows": len(df),
            "participants": df["id"].nunique() if "id" in df else None,
            "columns": df.shape[1],
            "rows by interval": by_interval,
        }
    )
    columns += [
        {"file": name, "column": c, "dtype": str(df[c].dtype), "% missing": pct_missing(df[c])}
        for c in df.columns
    ]
show("Files", pd.DataFrame(summary, dtype=object).set_index("file").fillna(""))
show("Columns", pd.DataFrame(columns).set_index(["file", "column"]))

## 2. Phase labels and hormones

In [ ]:
df = load(
    "hormones_and_selfreport", ["id", "day_in_study", "study_interval", "phase", *HORMONES], DATA
)
labeled = df.dropna(subset=["phase"])
tested = df.dropna(subset=[h for h in HORMONES if h in df], how="all")

show("Phase labels", categories(df, "phase"))
hormones = [{"hormone": h, **numeric(df[h])} for h in HORMONES if h in df]
show("Hormones (over all rows)", pd.DataFrame(hormones).set_index("hormone"))
show(
    "Days per participant (spread across participants)",
    pd.DataFrame(
        [
            {"days": "any row", **across_people(df.groupby("id")["day_in_study"].nunique())},
            {
                "days": "with a phase label",
                **across_people(labeled.groupby("id")["day_in_study"].nunique()),
            },
            {
                "days": "with a hormone test",
                **across_people(tested.groupby("id")["day_in_study"].nunique()),
            },
        ]
    ).set_index("days"),
)
by_interval = pd.Series(day_range(df, "day_in_study"), name="days").to_frame()
show("day_in_study by interval (p1 / p50 / p99)", by_interval)

## 3. Self-report (flow, symptoms)

In [ ]:
df = load("hormones_and_selfreport", ["id", *SELF_REPORT], DATA)
table, free_text = {}, []
for col in SELF_REPORT:
    if col in df:
        cats = categories(df, col)
        if "rows" in cats:
            table[col] = {v: str(r) for v, r in zip(cats["value"], cats["rows"], strict=True)}
        else:
            free_text.append(f"{col}: {cats['value'][0]}")
table = pd.DataFrame.from_dict(table, orient="index")
table = table[sorted(table.columns)].fillna("")
show("Rows per response (<5 ppl = fewer than 5 participants gave it)", table)
if free_text:
    print("\nNot shown (free text or too many values):", "; ".join(free_text))

## 4. Wearable signals and activity that line up with the Google Health API

In [ ]:
values, coverage, intervals, absent = [], [], {}, {}
for name, (day_col, value_cols) in SIGNALS.items():
    wanted = ["id", day_col, "study_interval", *value_cols]
    try:
        df = load(name, wanted, DATA)
    except FileNotFoundError:
        absent[name] = "file not found"
        continue
    missing_cols = [c for c in wanted if c not in df]
    if missing_cols:
        absent[name] = ", ".join(missing_cols)
    if name == "computed_temperature" and not missing_cols:
        # Mean nightly deviation from baseline, comparable to the API's nightly minus baseline.
        df["mean_dev_from_baseline"] = (
            df["baseline_relative_sample_sum"] / df["temperature_samples"]
        )
        value_cols = [*value_cols, "mean_dev_from_baseline"]
    values += [{"file": name, "column": c, **numeric(df[c])} for c in value_cols if c in df]
    if day_col in df and "id" in df:
        days = df.dropna(subset=[day_col])
        spread = across_people(days.groupby("id")[day_col].nunique())
        per_day = days.groupby(["id", day_col]).size()
        coverage.append(
            {
                "file": name,
                "participants": spread.pop("participants"),
                **{f"days {k}": v for k, v in spread.items()},
                "rows/day p50": per_day.median(),
                "rows/day p99": per_day.quantile(0.99),
            }
        )
        if "study_interval" in df:
            intervals[name] = day_range(df, day_col)

show("Values", pd.DataFrame(values).set_index(["file", "column"]))
show("Coverage: days with data per participant", pd.DataFrame(coverage).set_index("file"))
show("Day axis by interval (p1 / p50 / p99)", pd.DataFrame(intervals).T)
if absent:
    show("Expected but absent", pd.Series(absent, name="missing").to_frame())

## 5. Who is in the data (one column at a time, binned)

In [ ]:
df = load("subject-info", ["id", "birth_year", "age_of_first_menarche", *DEMOGRAPHICS], DATA)
if "birth_year" in df:
    age = 2022 - pd.to_numeric(df["birth_year"], errors="coerce")  # age at Interval 1
    df["age band (2022)"] = pd.cut(
        age, [0, 20, 25, 30, 35, 200], right=False, labels=["<20", "20-24", "25-29", "30-34", "35+"]
    )
if "age_of_first_menarche" in df:
    menarche = pd.to_numeric(df["age_of_first_menarche"], errors="coerce")
    df["age at first period"] = pd.cut(
        menarche, [0, 11, 13, 15, 100], right=False, labels=["<11", "11-12", "13-14", "15+"]
    )
for col in ["age band (2022)", "age at first period", *DEMOGRAPHICS]:
    if col in df:
        cats = categories(df, col)
        show(f"{col}: participants", cats[[c for c in ("value", "participants") if c in cats]])

## 6. Cleaning checks (evidence for `docs/data.md`, *Cleaning rules*)

Each sub-section backs one group of rules in the doc. If a number here moves, update the doc.

### 6a. Duplicate rows, and whether the copies agree

A duplicate group is two or more rows with the same key (`DUP_KEYS`). *% groups differing* ignores `last_modified_*`: 0% means exact copies, which are safe to drop.

In [ ]:
rows, by_column = [], []
for name, key in DUP_KEYS.items():
    df = try_load(name)
    if df is None:
        continue
    # Columns that should agree between copies. last_modified_* changes on every re-save.
    compare = [
        c
        for c in df
        if c not in (*key, "study_interval", "is_weekend") and not c.startswith("last_modified")
    ]
    for rnd, part in df.groupby("study_interval"):
        dup = part.duplicated(key, keep=False)
        groups = part[dup].groupby(key)
        copies = groups.size()
        differs = groups[compare].nunique(dropna=False).gt(1) if len(copies) else None
        differing = differs.any(axis=1) if len(copies) else copies
        if len(copies):
            share = differs.mean().mul(100).round(1)
            by_column += [
                {"file": name, "round": str(rnd), "column": c, "% groups differing": v}
                for c, v in share[share > 0].items()
            ]
        rows.append(
            {
                "file": name,
                "round": str(rnd),
                "rows": len(part),
                "rows in dup groups %": round(100 * dup.mean(), 1),
                "removed by dedup %": round(
                    100 * part.duplicated(key).mean(), 1
                ),  # all but one per key
                "dup groups": len(copies),
                "% groups differing": round(100 * differing.mean(), 1) if len(copies) else 0.0,
                "copies p50": copies.median() if len(copies) else "",
                "copies p99": copies.quantile(0.99) if len(copies) else "",
            }
        )
    del df
show("Duplicates by file and round", pd.DataFrame(rows).set_index(["file", "round"]))
if by_column:
    show(
        "Which columns differ between copies (share of duplicate groups)",
        pd.DataFrame(by_column).set_index(["file", "round", "column"]),
    )

### 6b. Other quirks: timestamps, reversed zone columns, sedentary, temperature zeros, exercise

In [ ]:
lengths = {}
for name, key in DUP_KEYS.items():
    time_col = next((c for c in key if "timestamp" in c), None)
    if time_col:
        t = try_load(name, [time_col])
        if t is not None:
            lengths[name] = ", ".join(
                str(n) for n in sorted(t[time_col].astype(str).str.len().unique())
            )
show(
    "Timestamp length in characters (8 = HH:MM:SS, time of day only)",
    pd.Series(lengths, name="lengths").to_frame(),
)

z = try_load("time_in_heart_rate_zones", ["study_interval", *ZONES])
if z is not None:
    show(
        "Heart-rate zones: median minutes/day by round (2024 columns read in reverse)",
        z.groupby("study_interval")[ZONES].median().round(0),
    )
    near_full_day = (z[ZONES].sum(axis=1) - 1440).abs() <= 5
    print("Share of days whose zones add up to ~1,440 min:", round(float(near_full_day.mean()), 3))

a = try_load("active_minutes", ["study_interval", "sedentary"])
if a is not None:
    show(
        "active_minutes.sedentary: % missing by round",
        a["sedentary"]
        .isna()
        .groupby(a["study_interval"])
        .mean()
        .mul(100)
        .round(1)
        .to_frame("% missing"),
    )

t = try_load("computed_temperature", ["study_interval", "baseline_relative_sample_sum"])
if t is not None:
    t = t.dropna(subset=["baseline_relative_sample_sum"])
    zero = t["baseline_relative_sample_sum"].eq(0)
    show(
        "computed_temperature: % of nights with deviation exactly 0",
        zero.groupby(t["study_interval"]).mean().mul(100).round(1).to_frame("% zero"),
    )

e = try_load("exercise", ["id", "logtype", "activityname"])
if e is not None:
    show("exercise: how sessions were recorded", categories(e, "logtype"))
    people = e.groupby("activityname")["id"].nunique()
    show(
        f"exercise: activity types with {MIN_CELL}+ participants",
        people[people >= MIN_CELL].sort_values(ascending=False).to_frame("participants"),
    )

### 6c. Why 2024 RHR is left out

Two ways the copies could have been recoverable: neighbouring days' values filed under one day (they would reappear on nearby days), or one copy position being the real series (it would change as little day to day as 2022 does).

In [ ]:
r = try_load("resting_heart_rate", ["id", "study_interval", "day_in_study", "value"])
if r is not None:
    r["position"] = r.groupby(["id", "day_in_study"]).cumcount() + 1  # copy's place in the file
    r = r[r["value"] > 0]
    r24 = r[r["study_interval"] == 2024]

    # 1. Do a day's 2024 values reappear on nearby days?
    values = r24.groupby(["id", "day_in_study"])["value"].apply(set)
    reuse = []
    for off in [1, 2, 3, 5, 7, 14]:
        before = shift_days(values, -off)  # day d now holds the set from day d - off
        common = values.index.intersection(before.index)
        seen = sum(len(values[i] & before[i]) for i in common)
        total = sum(len(values[i]) for i in common)
        reuse.append(
            {
                "days before": off,
                "days compared": len(common),
                "share of values seen then": round(seen / total, 3) if total else None,
            }
        )
    show(
        "RHR 2024: share of a day's values also present on an earlier day",
        pd.DataFrame(reuse).set_index("days before"),
    )

    # 2. Does any copy position change as little day to day as the 2022 series?
    def change_row(label, s):
        nxt = shift_days(s, 1)
        common = s.index.intersection(nxt.index)
        d = (s.loc[common] - nxt.loc[common]).abs()
        return {
            "series": label,
            "day pairs": len(d),
            "p50 change": round(float(d.median()), 1) if len(d) else None,
            "p90 change": round(float(d.quantile(0.9)), 1) if len(d) else None,
        }

    one_per_day = ["id", "day_in_study"]
    smooth = [
        change_row("2022", r[r["study_interval"] == 2022].groupby(one_per_day)["value"].first())
    ]
    for k in range(1, 6):
        copy_k = r24[r24["position"] == k].groupby(one_per_day)["value"].first()
        smooth.append(change_row(f"2024, copy {k}", copy_k))
    show(
        "RHR: day-to-day change in bpm (a real series moves ~1-2 bpm)",
        pd.DataFrame(smooth).set_index("series"),
    )

### 6d. Daily values by round

Exact copies dropped, then one value per participant-day using the rule in `docs/data.md`. A file is consistent across rounds when *median, in both rounds* is close for 2022 and 2024. RHR and steps are shown although 2024 is left out for them; steps under both duplicate rules.

In [ ]:
# file -> (day column, value column, how to make one value per day), as in docs/data.md
DAILY = {
    "resting_heart_rate": ("day_in_study", "value", "median"),
    "heart_rate_variability_details": ("day_in_study", "rmssd", "mean"),
    "respiratory_rate_summary": ("day_in_study", "full_sleep_breathing_rate", "median"),
    "computed_temperature": ("sleep_end_day_in_study", "nightly_temperature", "median"),
    "sleep": ("sleep_end_day_in_study", "minutesasleep", "max"),  # main sleep only
    "active_minutes": ("day_in_study", "lightly", "first"),
    "active_zone_minutes": ("day_in_study", "total_minutes", "sum"),
    "hormones_and_selfreport": ("day_in_study", "lh", "first"),
}
POSITIVE = {"resting_heart_rate", "respiratory_rate_summary"}  # 0 and -1 mean missing


def daily_values(name):
    """One value per (id, round, day): exact copies dropped, then the file's daily rule."""
    day_col, col, how = DAILY[name]
    extra = ["mainsleep"] if name == "sleep" else []
    df = try_load(
        name, list(dict.fromkeys(["study_interval", *DUP_KEYS[name], day_col, col, *extra]))
    )
    if df is None:
        return None
    df = df.drop_duplicates()
    if name in POSITIVE:
        df = df[df[col] > 0]
    if name == "sleep":
        df = df[df["mainsleep"]]
    return df.groupby(["id", "study_interval", day_col])[col].agg(how)


def by_round(label, daily):
    """Spread of daily values per round, plus the median for people in both rounds."""
    ids = daily.index.get_level_values("id")
    rounds = daily.index.get_level_values("study_interval")
    shared = set(ids[rounds == 2022]) & set(ids[rounds == 2024])
    rows = []
    for rnd, part in daily.groupby(level="study_interval"):
        people = part.index.get_level_values("id")
        row = {"signal": label, "round": str(rnd), "participants": suppress(people.nunique())}
        if people.nunique() >= MIN_CELL:
            row |= {f"p{q}": round(float(part.quantile(q / 100)), 2) for q in (25, 50, 75)}
            if len(shared) >= MIN_CELL:
                row["median, in both rounds"] = round(float(part[people.isin(shared)].median()), 2)
        rows.append(row)
    return rows


rows = []
for name, (_, col, how) in DAILY.items():
    daily = daily_values(name)
    if daily is not None:
        rows += by_round(f"{name}.{col} ({how}/day)", daily)

steps = try_load("steps", ["id", "study_interval", "day_in_study", "timestamp", "steps"])
if steps is not None:
    day_key = ["id", "study_interval", "day_in_study"]
    max_per_minute = steps.groupby([*day_key, "timestamp"])["steps"].max()
    rows += by_round("steps/day, max per minute", max_per_minute.groupby(level=day_key).sum())
    rows += by_round("steps/day, sum of all copies", steps.groupby(day_key)["steps"].sum())
    del steps, max_per_minute

show("Daily values by round", pd.DataFrame(rows).set_index(["signal", "round"]).fillna(""))